# Python建模库介绍
## 12.1 pandas与模型代码的接口

模型开发的通常工作流是使用pandas进行数据加载和清洗，然后切换到建模库进行建模。开发模型的重要一环是机器学习中的特征工程。它可以描述从原始数据集中提取信息的任何数据转换或分析，这些数据集可能在建模中有用。本书中学习的数据聚合和GroupBy工具常用于特征工程中。

利用特征工程提取出“好”特征超出了本书的范围，但我会尽量直白地介绍一些在数据操作和建模之间进行切换的方法。

pandas与其他分析库通常是靠NumPy的数组结合起来的。将DataFrame转换为NumPy数组，可以使用to_numpy方法：

In [36]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    'x0': [1, 2, 3, 4, 5],
    "x1": [0.01, -0.01, 0.25, -4.1, 0],
    'y': [-1.5, 0, 3.6, 1.3, -2]
})
data

,x0,x1,y
0,1,0.01,-1.5
1,2,-0.01,0.0
2,3,0.25,3.6
3,4,-4.10,1.3
4,5,0.00,-2.0


In [37]:
data.columns

Index(['x0', 'x1', 'y'], dtype='object')

In [38]:
data.to_numpy()

array([[ 1.  ,  0.01, -1.5 ],
       [ 2.  , -0.01,  0.  ],
       [ 3.  ,  0.25,  3.6 ],
       [ 4.  , -4.1 ,  1.3 ],
       [ 5.  ,  0.  , -2.  ]])

要转换回DataFrame，可以传入一个二维ndarray（可带有列名）：


In [39]:
df2 = pd.DataFrame(data.to_numpy(), columns=['a', 'b', 'c'])
df2

,a,b,c
0,1.0,0.01,-1.5
1,2.0,-0.01,0.0
2,3.0,0.25,3.6
3,4.0,-4.10,1.3
4,5.0,0.00,-2.0


to_numpy方法一般用于同构化数据。例如，数据全是数值类型。如果数据是异构化的，结果会是Python对象的ndarray：

In [40]:
df3 = data.copy()
df3['strings'] = ['a', 'b', 'c', 'd', 'e']
df3

,x0,x1,y,strings
0,1,0.01,-1.5,a
1,2,-0.01,0.0,b
2,3,0.25,3.6,c
3,4,-4.10,1.3,d
4,5,0.00,-2.0,e


In [41]:
df3.to_numpy()

array([[1, 0.01, -1.5, 'a'],
       [2, -0.01, 0.0, 'b'],
       [3, 0.25, 3.6, 'c'],
       [4, -4.1, 1.3, 'd'],
       [5, 0.0, -2.0, 'e']], dtype=object)

对于一些模型，你可能只想使用列的子集。我建议使用loc索引和to_numpy：

In [42]:
model_cols = ['x0', 'x1']
data.loc[:, model_cols].to_numpy()

array([[ 1.  ,  0.01],
       [ 2.  , -0.01],
       [ 3.  ,  0.25],
       [ 4.  , -4.1 ],
       [ 5.  ,  0.  ]])

一些库原生支持pandas，会自动完成这些工作：从DataFrame转换为NumPy，将模型参数名添加到输出表的列或Series上。对于其他情况，你可以手工进行“元数据管理”。

在7.5节中，我们学习了pandas的Categorical类型和pandas.get_dummies函数。假设数据集中有一个非数值列：

In [43]:
data['category'] = pd.Categorical(['a', 'b', 'a', 'a', 'b'])
data

,x0,x1,y,category
0,1,0.01,-1.5,a
1,2,-0.01,0.0,b
2,3,0.25,3.6,a
3,4,-4.10,1.3,a
4,5,0.00,-2.0,b


如果我们想将'category'列替换为虚拟变量，可以创建虚拟变量，删除'category'列，然后连接到结果中：

In [44]:
dummies = pd.get_dummies(data['category'], prefix='category')
dummies

,category_a,category_b
0,True,False
1,False,True
2,True,False
3,True,False
4,False,True


In [45]:
data_with_dummies = data.drop('category', axis=1).join(dummies)
data_with_dummies

,x0,x1,y,category_a,category_b
0,1,0.01,-1.5,True,False
1,2,-0.01,0.0,False,True
2,3,0.25,3.6,True,False
3,4,-4.10,1.3,True,False
4,5,0.00,-2.0,False,True


用虚拟变量拟合某些统计模型会有一些细微差别。当你不只有数值列时，使用Patsy（见12.2节）可能更简单，且更不容易出错。

## 12.2 用Patsy创建模型描述

Patsy(https://patsy.readthedocs.io/)是Python的一个库，它基于字符串的“公式语法”描述统计模型（尤其是线性模型），受到了R和S统计编程语言的公式语法的启发（但不完全一致）。安装statsmodels时，会自动安装Patsy：

`conda install statsmodels`

￼
Patsy适合描述statsmodels的线性模型，因此我会关注它的主要特点，让你尽快掌握。Patsy的公式是一个特殊的字符串语法，如下所示：
￼

`y ~ x0 + x1`

a+b不是将a与b相加的意思，而是为模型创建设计矩阵使用的术语。patsy.dmatrices函数接收一个公式字符串和一个数据集（可以是DataFrame或数组字典），为线性模型创建设计矩阵：

In [46]:
data = pd.DataFrame({
    'x0': [1, 2, 3, 4, 5],
    "x1": [0.01, -0.01, 0.25, -4.1, 0],
    'y': [-1.5, 0, 3.6, 1.3, -2]
})
data

,x0,x1,y
0,1,0.01,-1.5
1,2,-0.01,0.0
2,3,0.25,3.6
3,4,-4.10,1.3
4,5,0.00,-2.0


In [47]:
import patsy

y, X = patsy.dmatrices('y ~ x0 + x1', data)
y

DesignMatrix with shape (5, 1)
     y
  -1.5
   0.0
   3.6
   1.3
  -2.0
  Terms:
    'y' (column 0)

In [48]:
X

DesignMatrix with shape (5, 3)
  Intercept  x0     x1
          1   1   0.01
          1   2  -0.01
          1   3   0.25
          1   4  -4.10
          1   5   0.00
  Terms:
    'Intercept' (column 0)
    'x0' (column 1)
    'x1' (column 2)

这些Patsy的DesignMatrix实例是NumPy的ndarray，带有附加元数据：


In [49]:
np.asarray(y)

array([[-1.5],
       [ 0. ],
       [ 3.6],
       [ 1.3],
       [-2. ]])

In [50]:
np.asarray(X)

array([[ 1.  ,  1.  ,  0.01],
       [ 1.  ,  2.  , -0.01],
       [ 1.  ,  3.  ,  0.25],
       [ 1.  ,  4.  , -4.1 ],
       [ 1.  ,  5.  ,  0.  ]])

你可能想知道Intercept（截距）是从何而来的。这是线性模型（比如普通最小二乘回归）的惯例用法。添加+0到模型可以不显示截距：


In [52]:
patsy.dmatrices('y ~ x0 + x1 + 0', data)[1]

DesignMatrix with shape (5, 2)
  x0     x1
   1   0.01
   2  -0.01
   3   0.25
   4  -4.10
   5   0.00
  Terms:
    'x0' (column 0)
    'x1' (column 1)

Patsy对象可以直接传递给算法，比如numpy.linalg.lstsq，它执行普通最小二乘回归：

In [53]:
coef, resid, rank, s = np.linalg.lstsq(X, y, rcond=None)

In [54]:
coef

array([[ 0.31290976],
       [-0.07910564],
       [-0.26546384]])

模型的元数据保留在design_info属性中，因此你可以将模型列名重新附加到拟合系数上，以获得一个Series，例如：

In [56]:
coef = pd.Series(coef.squeeze(), index=X.design_info.column_names)
coef

Intercept    0.312910
x0          -0.079106
x1          -0.265464
dtype: float64

### 12.2.1 用Patsy公式进行数据转换

你可以将Python代码与patsy公式结合。在执行公式时，Patsy库将尝试在封闭作用域内查找使用的函数：

In [57]:
y, X = patsy.dmatrices('y ~ x0 + np.log(np.abs(x1) + 1)', data)
X

DesignMatrix with shape (5, 3)
  Intercept  x0  np.log(np.abs(x1) + 1)
          1   1                 0.00995
          1   2                 0.00995
          1   3                 0.22314
          1   4                 1.62924
          1   5                 0.00000
  Terms:
    'Intercept' (column 0)
    'x0' (column 1)
    'np.log(np.abs(x1) + 1)' (column 2)

常见的变量转换包括标准化（均值为0，方差为1）和居中（减去平均值）。Patsy有内置的函数进行此工作：

In [60]:
y, X = patsy.dmatrices('y ~ standardize(x0) + center(x1)', data)
X

DesignMatrix with shape (5, 3)
  Intercept  standardize(x0)  center(x1)
          1         -1.41421        0.78
          1         -0.70711        0.76
          1          0.00000        1.02
          1          0.70711       -3.33
          1          1.41421        0.77
  Terms:
    'Intercept' (column 0)
    'standardize(x0)' (column 1)
    'center(x1)' (column 2)

作为建模过程的一环，你可能将模型拟合到一个数据集，然后用另一个数据集来评估模型。另一个数据集可能是剩余的部分或是新数据。当执行居中和标准化等转换时，使用模型对新数据进行预测要格外小心。因为必须使用原始数据集的平均值或标准差等统计值来对新数据集做转换，所以也称作有状态转换。

patsy.build_design_matrices函数可以使用原始样本数据集的保存信息来转换样本外的新数据：

In [61]:
new_data = pd.DataFrame({
    'x0': [6, 7, 8, 9],
    "x1": [3.1, -0.5, 0.0, 2.3],
    'y': [1, 2, 3, 4]
    })
new_data

,x0,x1,y
0,6,3.1,1
1,7,-0.5,2
2,8,0.0,3
3,9,2.3,4


In [62]:
new_x = patsy.build_design_matrices([X.design_info], new_data)
new_x

[DesignMatrix with shape (4, 3)
   Intercept  standardize(x0)  center(x1)
           1          2.12132        3.87
           1          2.82843        0.27
           1          3.53553        0.77
           1          4.24264        3.07
   Terms:
     'Intercept' (column 0)
     'standardize(x0)' (column 1)
     'center(x1)' (column 2)]

加号(+)在Patsy的上下文中不表示加法，当你按照名称将数据集的列相加时，必须用特殊的函数I将列名封装起来：

In [63]:
y, X = patsy.dmatrices('y ~ I(x0 + x1)', data)
X

DesignMatrix with shape (5, 2)
  Intercept  I(x0 + x1)
          1        1.01
          1        1.99
          1        3.25
          1       -0.10
          1        5.00
  Terms:
    'Intercept' (column 0)
    'I(x0 + x1)' (column 1)

### 12.2.2 分类数据和Patsy

当你在Patsy公式中使用非数值数据时，会默认将其转换为虚拟变量。如果有截距，会去掉其中一个，以避免共线性：

In [64]:
data = pd.DataFrame({
    'key1': ['a', 'a', 'b', 'b', 'a', 'b', 'a', 'b'],
    'key2': [0, 1, 0, 1, 0, 1, 0, 0],
    'v1': [1, 2, 3, 4, 5, 6, 7, 8],
    'v2': [-1, 0, 2.5, -0.5, 4, -1.2, 0.2, -1.7]
    })
data

,key1,key2,v1,v2
0,a,0,1,-1.0
1,a,1,2,0.0
2,b,0,3,2.5
3,b,1,4,-0.5
4,a,0,5,4.0
5,b,1,6,-1.2
6,a,0,7,0.2
7,b,0,8,-1.7


In [65]:
y, X = patsy.dmatrices('v2 ~ key1', data)
X

DesignMatrix with shape (8, 2)
  Intercept  key1[T.b]
          1          0
          1          0
          1          1
          1          1
          1          0
          1          1
          1          0
          1          1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)

如果你从模型中忽略截距，则每个分类值的列都会包含在模型设计矩阵中：

In [66]:
y, X = patsy.dmatrices('v2 ~ key1 + 0', data)
X

DesignMatrix with shape (8, 2)
  key1[a]  key1[b]
        1        0
        1        0
        0        1
        0        1
        1        0
        0        1
        1        0
        0        1
  Terms:
    'key1' (columns 0:2)

使用C函数，可以将数值列解释为分类类型：

In [68]:
y, X = patsy.dmatrices('v2 ~ C(key2)', data)
X

DesignMatrix with shape (8, 2)
  Intercept  C(key2)[T.1]
          1             0
          1             1
          1             0
          1             1
          1             0
          1             1
          1             0
          1             0
  Terms:
    'Intercept' (column 0)
    'C(key2)' (column 1)

当你在模型中使用多个分类项时，事情就会变复杂，因为会包括key1:key2形式的交互项，它可以用在方差分析(ANOVA)模型中：


In [69]:
data['key2'] = data['key2'].map({0: 'zero', 1: 'one'})
data

,key1,key2,v1,v2
0,a,zero,1,-1.0
1,a,one,2,0.0
2,b,zero,3,2.5
3,b,one,4,-0.5
4,a,zero,5,4.0
5,b,one,6,-1.2
6,a,zero,7,0.2
7,b,zero,8,-1.7


In [70]:
y, X = patsy.dmatrices('v2 ~ key1 + key2', data)
X

DesignMatrix with shape (8, 3)
  Intercept  key1[T.b]  key2[T.zero]
          1          0             1
          1          0             0
          1          1             1
          1          1             0
          1          0             1
          1          1             0
          1          0             1
          1          1             1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)

In [71]:
y, X = patsy.dmatrices('v2 ~ key1 + key2 + key1:key2', data)
X

DesignMatrix with shape (8, 4)
  Intercept  key1[T.b]  key2[T.zero]  key1[T.b]:key2[T.zero]
          1          0             1                       0
          1          0             0                       0
          1          1             1                       1
          1          1             0                       0
          1          0             1                       0
          1          1             0                       0
          1          0             1                       0
          1          1             1                       1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)
    'key1:key2' (column 3)

## 12.3 statsmodels介绍

statsmodels(https://www.statsmodels.org)是Python用于拟合多种统计模型、进行统计试验、数据探索和可视化的库。statsmodels包含许多经典的频率论统计方法，可以在其他库中找到贝叶斯方法和机器学习模型。

statsmodels包含以下模型：
- 线性模型，包括广义线性模型和鲁棒线性模型
- 线性混合效应模型
- 方差分析方法
- 时间序列过程和状态空间模型
- 广义矩估

### 12.3.1 对线性模型进行估计

statsmodels有多种线性回归模型，包括从基本（例如，普通最小二乘）到复杂（例如，迭代加权最小二乘）的模型。

statsmodels的线性模型有两种不同的接口：基于数组和基于公式。它们可以通过对应的API模块导入来访问：

In [72]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

为了展示使用方法，我们从一些随机数据生成线性模型。在Jupyter代码框中运行以下代码：

In [73]:
rng = np.random.default_rng(seed=12345)

def dnorm(mean, variance, size=1):
    if isinstance(size, int):
        size = (size,)
    return mean + np.sqrt(variance) * rng.standard_normal(size=size)

N = 100
X = np.c_[dnorm(0, 0.4, size=N),
          dnorm(0, 0.6, size=N),
          dnorm(0, 0.2, size=N)]
eps = dnorm(0, 0.1, size=N)
beta = [0.1, 0.3, 0.5]
y = np.dot(X, beta) + eps

这里，我使用了“真实”模型和已知参数beta。此时，dnorm是辅助函数，用于生成具有特定的均值和方差的正态分布数据。现在有

In [74]:
X[:5]

array([[-0.90050602, -0.18942958, -1.0278702 ],
       [ 0.79925205, -1.54598388, -0.32739708],
       [-0.55065483, -0.12025429,  0.32935899],
       [-0.16391555,  0.82403985,  0.20827485],
       [-0.04765129, -0.21314698, -0.04824364]])

In [75]:
y[:5]

array([-0.59952668, -0.58845445,  0.18563386, -0.00747657, -0.01537445])

像之前在Patsy中看到的，线性模型通常要和一个截距项拟合。sm.add_constant函数可以添加一个截距列到现有的矩阵：

In [76]:
X_model = sm.add_constant(X)
X_model[:5]

array([[ 1.        , -0.90050602, -0.18942958, -1.0278702 ],
       [ 1.        ,  0.79925205, -1.54598388, -0.32739708],
       [ 1.        , -0.55065483, -0.12025429,  0.32935899],
       [ 1.        , -0.16391555,  0.82403985,  0.20827485],
       [ 1.        , -0.04765129, -0.21314698, -0.04824364]])

sm.OLS类可以拟合普通最小二乘回归：


In [79]:
model = sm.OLS(y, X_model)
results = model.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.470
Model:                            OLS   Adj. R-squared:                  0.453
Method:                 Least Squares   F-statistic:                     28.36
Date:                Mon, 01 Dec 2025   Prob (F-statistic):           3.23e-13
Time:                        22:23:50   Log-Likelihood:                -25.390
No. Observations:                 100   AIC:                             58.78
Df Residuals:                      96   BIC:                             69.20
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0208      0.032     -0.653      0.516      -0.084       0.042
x1             0.0658      0.054      1.220      0.226      -0.041       0.173
x2             0.2690      0.043      6.312      0.000       0.184       0.354
x3             0.4494      0.068      6.567      0.000       0.314       0.585
==============================================================================
Omnibus:                        0.429   Durbin-Watson:                   1.878
Prob(Omnibus):                  0.807   Jarque-Bera (JB):                0.296
Skew:                           0.133   Prob(JB):                        0.863
Kurtosis:                       2.995   Cond. No.                         2.16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

这里的参数名为通用名x1、x2等。假设所有模型参数都在一个DataFrame中：

In [80]:
data = pd.DataFrame(X, columns=['col0', 'col1', 'col2'])
data['y'] = y
data

,col0,col1,col2,y
0,-0.900506,-0.189430,-1.027870,-0.599527
1,0.799252,-1.545984,-0.327397,-0.588454
2,-0.550655,-0.120254,0.329359,0.185634
3,-0.163916,0.824040,0.208275,-0.007477
4,-0.047651,-0.213147,-0.048244,-0.015374
...,...,...,...,...
95,-0.039152,0.531515,-0.587640,-0.067934
96,-0.227355,0.941139,-0.228237,0.831554
97,-0.473484,0.167359,-0.044659,0.070316
98,-0.610622,-0.747349,-0.057917,-0.386481


In [81]:
results = smf.ols('y ~ col0 + col1 + col2', data=data).fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.470
Model:                            OLS   Adj. R-squared:                  0.453
Method:                 Least Squares   F-statistic:                     28.36
Date:                Mon, 01 Dec 2025   Prob (F-statistic):           3.23e-13
Time:                        22:27:40   Log-Likelihood:                -25.390
No. Observations:                 100   AIC:                             58.78
Df Residuals:                      96   BIC:                             69.20
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0208      0.032     -0.653      0.516      -0.084       0.042
col0           0.0658      0.054      1.220      0.226      -0.041       0.173
col1           0.2690      0.043      6.312      0.000       0.184       0.354
col2           0.4494      0.068      6.567      0.000       0.314       0.585
==============================================================================
Omnibus:                        0.429   Durbin-Watson:                   1.878
Prob(Omnibus):                  0.807   Jarque-Bera (JB):                0.296
Skew:                           0.133   Prob(JB):                        0.863
Kurtosis:                       2.995   Cond. No.                         2.16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

观察statsmodels是如何返回Series结果的，它附带DataFrame的列名。当使用公式和pandas对象时，我们不需要使用add_constant。

给出一个样本外数据，你可以根据估计的模型参数计算预测值：

In [83]:
results.predict(data[:5])

0   -0.592959
1   -0.531160
2    0.058636
3    0.283658
4   -0.102947
dtype: float64

### 12.3.2 对时间序列过程进行估计

statsmodels的另一类模型是对时间序列进行分析，包括自回归过程、卡尔曼滤波和其他状态空间模型，以及多变量自回归模型。

让我们用自回归结构和噪声来模拟一些时间序列数据。在Jupyter中运行以下命令：

In [84]:
init_x = 4
values = [init_x, init_x]
N = 1000
b0 = 0.8
b1 = -0.4
noise = dnorm(0, 0.1, N)

for i in range(N):
    new_x = values[-1] * b0 + values[-2] * b1 + noise[i]
    values.append(new_x)

这个数据具有AR(2)结构（两个滞后），参数是0.8和-0.4。当拟合AR模型时，你可能不知道要包含的滞后项的个数，因此可以用更大的滞后数来拟合这个模型：

In [85]:
from statsmodels.tsa.ar_model import AutoReg
MAXLAGS = 5
model = AutoReg(y, lags=MAXLAGS)
results = model.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            AutoReg Model Results                             
==============================================================================
Dep. Variable:                      y   No. Observations:                  100
Model:                     AutoReg(5)   Log Likelihood                 -52.425
Method:               Conditional MLE   S.D. of innovations              0.420
Date:                Mon, 01 Dec 2025   AIC                            118.851
Time:                        22:34:52   BIC                            136.728
Sample:                             5   HQIC                           126.074
                                  100                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0095      0.043     -0.219      0.827      -0.094       0.075
y.L1          -0.0529      0.102     -0.520      0.603      -0.253       0.147
y.L2           0.0574      0.101      0.566      0.572      -0.141       0.256
y.L3          -0.0618      0.101     -0.609      0.543      -0.261       0.137
y.L4          -0.1521      0.102     -1.485      0.138      -0.353       0.049
y.L5           0.1273      0.102      1.245      0.213      -0.073       0.328
                                    Roots                                    
=============================================================================
                  Real          Imaginary           Modulus         Frequency
-----------------------------------------------------------------------------
AR.1           -1.0848           -0.7913j            1.3428           -0.3997
AR.2           -1.0848           +0.7913j            1.3428            0.3997
AR.3            0.7130           -1.3187j            1.4991           -0.1711
AR.4            0.7130           +1.3187j            1.4991            0.1711
AR.5            1.9386           -0.0000j            1.9386           -0.0000
-----------------------------------------------------------------------------
"""

## 12.4 scikit-learn介绍

scikit-learn(https://scikit-learn.org)是使用广泛、用途多样的Python机器学习库之一。它包含多种标准的监督机器学习和非监督机器学习方法，以及模型选择和评估、数据转换、数据加载和模型持久化工具。这些模型可以用于分类、聚类、预测和其他常见任务。

机器学习方面的学习和应用scikit-learn解决实际问题的线上和纸质资料很多。本节，我会简要介绍scikit-learn API的风格。

近些年来，pandas和scikit-learn的融合取得了重大进展，读者读到本书时，两者之间应该有了进一步的融合。建议读者查阅最新的项目文档。

作为本章的示例，我使用了Kaggle竞赛(https://www.kaggle.com/c/titanic)的经典数据集——泰坦尼克号在1912年沉没时船上乘客的生还率。我们用pandas加载训练数据集和测试数据集：


In [86]:
train = pd.read_csv('datasets/titanic/train.csv')
test = pd.read_csv('datasets/titanic/test.csv')

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


statsmodels和scikit-learn通常不能接收缺失数据，因此我们要查看列是否包含缺失值：

In [87]:
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [88]:
test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

在像这样的统计和机器学习示例中，根据数据中的特征，一个典型的任务是预测乘客能否生还。模型先在训练数据集中拟合，然后用样本外测试数据集进行评估。

我想用Age作为预测值，但是它包含缺失值。有多种补全缺失数据的方法，我用的是一种简单方法，用训练数据集的中位数补全两个表的空值：

In [90]:
impute_value = train['Age'].median()
train['Age'] = train['Age'].fillna(impute_value)
test['Age'] = test['Age'].fillna(impute_value)

现在需要指定模型。我增加了一个列IsFemale作为'Sex'列的编码：


In [91]:
train['IsFemale'] = (train['Sex'] == 'female').astype(int)
test['IsFemale'] = (test['Sex'] == 'female').astype(int)

然后，我们确定一些模型变量，并创建NumPy数组：

In [92]:
predictors = ['Pclass', 'Age', 'IsFemale']
X_train = train[predictors].to_numpy()
X_test = test[predictors].to_numpy()
y_train = train['Survived'].to_numpy()

X_train[:5]

array([[ 3., 22.,  0.],
       [ 1., 38.,  1.],
       [ 3., 26.,  1.],
       [ 1., 35.,  1.],
       [ 3., 35.,  0.]])

In [93]:
y_train[:5]

array([0, 1, 1, 1, 0])

不能保证这是一个好模型，也不能保证这些特征得到了合适的处理。我们使用scikit-learn的LogisticRegression模型创建一个模型实例：


In [94]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

使用模型的fit方法，将模型拟合到训练数据：


In [95]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


现在，我们可以用model.predict在测试数据上进行预测：

In [96]:
y_predict = model.predict(X_test)
y_predict

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

如果你有测试数据集的真实值，可以计算准确率或其他误差度指标：

In [105]:
# (y_true == y_predict).mean()

在实际中，模型训练经常有许多额外的复杂因素。许多模型有可以调节的参数，有些方法（比如交叉验证）可以用来进行参数调节，避免对训练数据过拟合。这通常可以对新数据提高预测表现或健壮性。

交叉验证通过分割训练数据来模拟样本外预测。基于模型的准确度分数（比如均方差），可以对模型参数进行网格搜索。有些模型，如logistic回归，有内置的交叉验证的估计类。例如，LogisticRegressionCV类可以用一个参数指定对模型正则化参数C的网格搜索粒度：

In [106]:
from sklearn.linear_model import LogisticRegressionCV
model_cv = LogisticRegressionCV(Cs=10)
model_cv.fit(X_train, y_train)

,Cs,10
,fit_intercept,True
,cv,None
,dual,False
,penalty,'l2'
,scoring,None
,solver,'lbfgs'
,tol,0.0001
,max_iter,100
,class_weight,None
,n_jobs,None


要手动进行交叉验证，你可以使用辅助函数cross_val_score，它可以处理数据分割过程。例如，要使用训练数据的4个非重叠部分来交叉验证我们的模型，可以这样做：

In [107]:
from sklearn.model_selection import cross_val_score
model = LogisticRegression(C=10)
scores = cross_val_score(model, X_train, y_train, cv=4)
scores

array([0.77578475, 0.79820628, 0.77578475, 0.78828829])

默认的评分指标取决于模型本身，但是可以显式指定一个评分函数。交叉验证过的模型需要更长时间来训练，但会有更好的模型性能。